In [ ]:
import scipy as sp
import numpy as np
import thewalrus as wr
import functools
import matplotlib.pyplot as plt
import ipywidgets as widgets

In [ ]:
def T(n):
    """Defines conversion matrix for thewalrus
    T.T@sigma@T is our converted matrix
    T@sigma@T.T converts back
    """
    v1 = np.array([[1,0]])
    v2 = np.array([[0,1]])
    T1 = sp.linalg.block_diag(*([v1]*n))
    T2 = sp.linalg.block_diag(*([v2]*n))
    T = np.block([[T1],[T2]])
    return T


In [ ]:
T(2)

In [ ]:
def σ(η,ns,nb):
    return np.array([
        [1 +2*η*ns + 2*nb,0,-2*np.sqrt(ns*(1+ns)*η),0],
        [0,1 +2*η*ns + 2*nb,0,2*np.sqrt(ns*(1+ns)*η)],
        [-2*np.sqrt(ns*(1+ns)*η),0,1 +2*ns,0],
        [0,2*np.sqrt(ns*(1+ns)*η),0,1 +2*ns]
    ])
def cov(η,ns,nb):
    return T(2)@σ(η,ns,nb)@T(2)
@functools.cache
def Ps_pn(η,ns,nb,maxval=10):
    means = np.zeros(4)
    covval = cov(η,ns,nb)
    Ps = wr.quantum.probabilities(means,covval,maxval,atol=1e-10,rtol=1e-7)
    return Ps
def Ps_pnij(η,ns,nb,i,j):
    return Ps_pn(η,ns,nb)[i][j]
vec_Ps_pn = np.vectorize(Ps_pnij)
def to_diff_pn(ηs,*args):
    ns = args[0]
    nb = args[1]
    i = args[2]
    j = args[3]
    return vec_Ps_pn(ηs,ns,nb,i,j)
def FI_pn(ηs,nss,nbs):
    indices = np.arange(10)
    ηgrid,nsgrid,nbgrid,igrid,jgrid = np.meshgrid(ηs,nss,nbs,indices,indices,indexing='ij')
    Ps = vec_Ps_pn(ηgrid,nsgrid,nbgrid,igrid,jgrid)
    ds = np.zeros((len(ηs),len(nss),len(nbs),10,10))
    for i in range(10):
        for j in range(10):
            for k,ns in enumerate(nss):
                for l,nb in enumerate(nbs):
                    derivres = sp.differentiate.derivative(to_diff_pn,ηvals,args = [nsval,nbval,i,j],initial_step=1e-5)
                    ds[:,k,l,i,j] = derivres.df
    presum = ds**2/Ps
    presum[~np.isfinite(presum)] = 0
    FI = np.sum(presum,axis=(3,4))
    return FI


def σsu(η,ns,nb):
    bkrd = 1+2*ns*((1+ns)*(1+np.sqrt(η))**2+nb)
    corr = 2*np.sqrt(ns*(1+ns))*(1+np.sqrt(η)+nb+ns*(1+np.sqrt(η))**2)
    return np.array([
        [bkrd + 2*nb,0,-corr,0],
        [0,bkrd+ 2*nb,0,corr],
        [-corr,0,bkrd+2*ns*(1-η),0],
        [0,corr,0,bkrd+2*ns*(1-η)]
    ])
def covsu(η,ns,nb):
    return T(2)@σsu(η,ns,nb)@T(2)
@functools.cache
def Ps_su(η,ns,nb,maxval=10):
    means = np.zeros(4)
    covval = covsu(η,ns,nb)
    Ps = wr.quantum.probabilities(means,covval,maxval,atol=1e-10,rtol=1e-7)
    return Ps
def Ps_suij(η,ns,nb,i,j):
    return Ps_su(η,ns,nb)[i][j]
vec_Ps_su = np.vectorize(Ps_suij)
def to_diff_su(ηs,*args):
    ns = args[0]
    nb = args[1]
    i = args[2]
    j = args[3]
    return vec_Ps_su(ηs,ns,nb,i,j)
def FI_su(ηs,nss,nbs):
    indices = np.arange(10)
    ηgrid,nsgrid,nbgrid,igrid,jgrid = np.meshgrid(ηs,nss,nbs,indices,indices,indexing='ij')
    Ps = vec_Ps_su(ηgrid,nsgrid,nbgrid,igrid,jgrid)
    ds = np.zeros((len(ηs),len(nss),len(nbs),10,10))
    for i in range(10):
        for j in range(10):
            for k,ns in enumerate(nss):
                for l,nb in enumerate(nbs):
                    derivres = sp.differentiate.derivative(to_diff_su,ηvals,args = [nsval,nbval,i,j],initial_step=1e-5)
                    ds[:,k,l,i,j] = derivres.df
    presum = ds**2/Ps
    presum[~np.isfinite(presum)] = 0
    FI = np.sum(presum,axis=(3,4))
    return FI

In [ ]:
ηval = .7
nsval = 1
nbval = .01
Tval = T(2)
σval = σ(ηval,nsval,nbval)
C,S = wr.decompositions.williamson(Tval.T@σval@Tval)
ν1 = (1-ηval)*nsval-nbval + np.sqrt( ((1-ηval)*nsval + (1+nbval))**2 + 4*ηval*nbval*nsval)
ν2 = -(1-ηval)*nsval+nbval + np.sqrt( ((1-ηval)*nsval + (1+nbval))**2 + 4*ηval*nbval*nsval)
λ = np.sqrt( (1+nbval+nsval+ηval*nsval +2*np.sqrt(ηval*nsval*(1+nsval)))/np.sqrt( (1+nbval+nsval-ηval*nsval)**2+4*nsval*nbval*ηval))
O,D,Q = wr.decompositions.blochmessiah(S)

In [ ]:
display(σval)
display(np.round(Tval@O@D@Q@C@Q.T@D@O.T@Tval,3))

In [ ]:
np.round(Tval@D@Tval.T,3)

In [ ]:
λ

In [ ]:
display(Tval@σval@Tval)

In [ ]:
σval

In [ ]:
ηval = .99
nsval = .0001
nbval = .01
Ps_pn(ηval,nsval,nbval)

In [ ]:
ηvals = np.linspace(1e-3,1-1e-3,80,dtype=np.double)
nsvals = np.logspace(-4,.5,10,dtype=np.double)
nbvals = np.logspace(-5,-.5,10,dtype=np.double)
df = sp.differentiate.derivative(to_diff_pn,[ηval],args = [nsval,nbval,1,1],initial_step=1e-5)

In [ ]:
df.df

In [ ]:
FIspn = FI_pn(ηvals,nsvals,nbvals)
FIssu = FI_su(ηvals,nsvals,nbvals)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot()
ax.plot(ηvals,FIspn[:,4,8],label="Biphoton")
ax.plot(ηvals,FIssu[:,4,8],label="SU(1,1)")
ax.set_xlabel('eta')
ax.set_ylabel('FI')
plt.legend(loc='best')
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot()
ax.plot(ηvals,FIssu[:,4,8]-FIspn[:,4,8],label="Difference")
ax.set_xlabel('eta')
ax.set_ylabel('FI')
plt.legend(loc='best')
plt.show()

In [ ]:
nsslider = widgets.IntSlider(
    value=7,
    min=0,
    max=9,
    step=1,
    description='Ns index',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
nbslider = widgets.IntSlider(
    value=7,
    min=0,
    max=9,
    step=1,
    description='Nb index:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

In [ ]:
def display_graph(nsi,nbi):
    fig = plt.figure()
    ax = fig.add_subplot()
    ax.plot(ηvals,FIssu[:,nsi,nbi]-FIspn[:,nsi,nbi],label="Difference")
    ax.set_xlabel('eta')
    ax.set_ylabel('FI')
    plt.legend(loc='best')
    return fig


In [ ]:
widgets.interact(display_graph,nsi = nsslider,nbi = nbslider,continuous=False)